In [46]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [53]:
import duckdb
import pandas as pd

from icare_risk.clinphen import DuckDBSource
from icare_risk.clinphen.config.schema import load_schema_from_yaml
from icare_risk.utils.paths import get_path

# Define path to your schema configuration
schema_cfg = get_path('configs/schema.yaml')

# Load schema and initialize DuckDB source
schema = load_schema_from_yaml(schema_cfg)
source = DuckDBSource(connection=duckdb.connect())

print("✅ DuckDB source and schema loaded successfully.")

✅ Schema validation passed! All files exist and all physical columns match successfully.
✅ DuckDB source and schema loaded successfully.


In [54]:
# --------------------------------------------------------
# Step 2: Test 1 - Column projection (memory optimization)
# --------------------------------------------------------
# To prevent Out-Of-Memory crashes on an 8GB RAM machine, you can 
# restrict the load to only the specific columns you need

# Request only subject, timestamp, code, and value columns for microbiology
df_projected = source.load_domain(
    schema.domains["microbiology"],
    subjects=['10001', '10002'],
    columns=["code", "organism", "sensitivity"]
)

# Check total memory footprint using deep inspection
total_bytes = df_projected.memory_usage(deep=True).sum()
total_mb = total_bytes / (1024 ** 2)

print(f"📊 Projected DataFrame Shape: {df_projected.shape}")
print(f"💾 Total Memory Usage: {total_mb:.2f} MB")
display(df_projected.head())

📊 Projected DataFrame Shape: (3, 5)
💾 Total Memory Usage: 0.00 MB


,subject,timestamp,code,organism,sensitivity
0,10001,2022-11-08,zz01,escherichia coli,ertapenem : 0.047
1,10002,2023-06-25,zz01,coliform (second type),augmentin : sensitive
2,10002,2022-11-26,zz00,staphylococcus aureus,linezolid : susceptible


In [55]:
# --------------------------------------------------------
# Step 3: Test 2 - Column projection (memory optimization)
# --------------------------------------------------------
# Filter microbiology records directly on disk by specific 
# sample sites and resistance statuses.

df_filtered = source.load_domain(
    schema.domains["microbiology"],
    subjects=['10001', '10002', '10003'],
    columns=["code", "site", "organism", "sensitivity"],
    filters={
        "site": ["rectal swab"]     # Filter string content on site column
    }
)

print(f"🔍 Filtered Result Shape: {df_filtered.shape[0]:,} rows × {df_filtered.shape[1]} columns")
print(f"💾 Filtered Memory Footprint: {df_filtered.memory_usage(deep=True).sum() / (1024**2):.2f} MB")
display(df_filtered.head(10))

🔍 Filtered Result Shape: 1 rows × 6 columns
💾 Filtered Memory Footprint: 0.00 MB


,subject,timestamp,code,site,organism,sensitivity
0,10001,2022-11-08,zz01,rectal swab,escherichia coli,ertapenem : 0.047


In [56]:
# --------------------------------------------------------
# Step 4: Test 3 - Column projection (memory optimization)
# --------------------------------------------------------
# Run this quick comparison snippet to verify how much memory 
# and overhead are saved when combining column projection and 
# value filtering versus loading raw tables:
import time

# Benchmark 1: Unfiltered / Wide load simulation
t0 = time.perf_counter()
df_raw_check = source.load_domain(
    schema.domains["microbiology"], 
    subjects=['10001', '10002']
)
time_raw = time.perf_counter() - t0
mem_raw = df_raw_check.memory_usage(deep=True).sum() / (1024 ** 2)

# Benchmark 2: Optimized & Filtered load
t0 = time.perf_counter()
df_opt_check = source.load_domain(
    schema.domains["microbiology"],
    subjects=['10001', '10002'],
    columns=["code", "organism"],
    filters={"site": ["URINE"]}
)
time_opt = time.perf_counter() - t0
mem_opt = df_opt_check.memory_usage(deep=True).sum() / (1024 ** 2)

print("--- Performance & Memory Benchmark ---")
print(f"Raw Load      -> Time: {time_raw:.4f}s | Memory: {mem_raw:.2f} MB")
print(f"Optimized Load -> Time: {time_opt:.4f}s | Memory: {mem_opt:.2f} MB")
print(f"✨ Memory Saved: {mem_raw - mem_opt:.2f} MB")

--- Performance & Memory Benchmark ---
Raw Load      -> Time: 0.0063s | Memory: 0.00 MB
Optimized Load -> Time: 0.0032s | Memory: 0.00 MB
✨ Memory Saved: 0.00 MB


In [57]:
# --------------------------------------------------------
# Step 5: Test 4 - Completely manual no need of yaml
# --------------------------------------------------------
# Run this quick manual example.

# Libraries
import duckdb
from icare_risk.clinphen import DuckDBSource
from icare_risk.clinphen.config.schema import TableSchema
from icare_risk.utils.paths import get_path

# Define path to your schema configuration
filepath = get_path('data/synthetic/1000000p/icare_vital_signs_anon.parquet')
filepath = get_path('data/synthetic/1000000p/icare_pharmacy_prescribing_anon.parquet')

# Initialize DuckDB source
source = DuckDBSource(connection=duckdb.connect())

# Define schema programmatically on the fly
manual_vitals_schema = TableSchema(
    source=str(filepath),
    subject="SUBJECT",
    timestamp="ORDER_DT_TM",
    mapping={
        "code": "MEDICATION_NAME_SHORT",
        "name": "MEDICATION_NAME_SHORT",
        "value": "ORDERED_DOSE",
        "THERAPEUTICAL_CLASS": "THERAPEUTICAL_CLASS"
    }
)

# Load data using the manual schema
df_manual = source.load_domain(
    table_schema=manual_vitals_schema,
    #subjects=['10001', '10002'],
    codes=['Amoxicillin']
)

display(df_manual.head())

,subject,timestamp,code,name,value,THERAPEUTICAL_CLASS
0,908118,2021-04-27,Amoxicillin,Amoxicillin,353.00,Antibiotics
1,908118,2021-06-12,Amoxicillin,Amoxicillin,144.93,Antibiotics
2,908118,2023-10-13,Amoxicillin,Amoxicillin,302.09,Antibiotics
3,908118,2022-03-29,Amoxicillin,Amoxicillin,962.86,Antibiotics
4,908119,2020-12-14,Amoxicillin,Amoxicillin,155.59,Antibiotics


In [65]:
# ----------------------------------------------------------
# Step 6: Test 5 - Completely automated from phenotypes.yaml
# ----------------------------------------------------------
# Run this quick automated example.
import duckdb
from icare_risk.clinphen import DuckDBSource
from icare_risk.clinphen.config.schema import load_schema_from_yaml
from icare_risk.clinphen.registry.registry import load_phenotypes_from_yaml
from icare_risk.utils.paths import get_path

# 1. Define configuration paths
schema_cfg = get_path('configs/schema.yaml')
phenotype_cfg = get_path('configs/phenotypes.yaml')

# 2. Load schema and phenotype registry
schema = load_schema_from_yaml(schema_cfg)
registry = load_phenotypes_from_yaml(phenotype_cfg)
source = DuckDBSource(connection=duckdb.connect())

# 3. Extract global dependencies automatically from phenotypes.yaml
required_codes, required_cols = registry.get_global_dependencies()

print(required_codes['diagnosis'])

# 4. Load domains dynamically using extracted codes and columns
df_problems = source.load_domain(
    schema.domains["problems"],
    codes=required_codes.get("problems"),
    columns=None,
    skip_mapping=True
)

df_diagnosis = source.load_domain(
    schema.domains["diagnosis"]
)

print(f"Loaded problems shape: {df_problems.shape}")
print(f"Loaded diagnosis shape: {df_diagnosis.shape}")
display(df_problems.head(3))
display(df_diagnosis.head(3))

✅ Schema validation passed! All files exist and all physical columns match successfully.
Skipping 'definitions': missing module or function definition.
{'O98.7', 'ZV01A00', '43C3.11', '165816005', '161513008', 'Z21', '19030005', '442662004', '89293008', '1142045004', '43C3.00', 'R109.00', 'R75'}
Loaded problems shape: (2293, 6)
Loaded diagnosis shape: (29962, 8)


,SUBJECT,ENCNTR_ID,PROBLEM_CODE,PROBLEM_DESC,PROBLEM_DT_TM,UPDATE_DT_TM
0,10004,10004,19030005,hiv,2020-06-20,2020-07-07
1,10006,10006,266257000,tia,2020-08-09,2020-09-22
2,10010,10010,118599009,"lymphoma, hodgkins",2022-06-13,2022-09-10


,subject,timestamp,code,name,DIAGNOSIS_CODE_ICD,DIAGNOSIS_DESC_ICD,DIAGNOSIS_CODE_SNOMED,DIAGNOSIS_DESC_SNOMED
0,10001,2022-11-26,j123,human metapneumovirus pneumonia,j123,human metapneumovirus pneumonia,10625271000119109,NOT IN ENUMERATIONS LIST
1,10001,2023-10-27,r298,other and unspecified symptoms and signs invol...,r298,other and unspecified symptoms and signs invol...,437986008,NOT IN ENUMERATIONS LIST
2,10001,2020-05-13,none,none,none,none,1052951000000105,NOT IN ENUMERATIONS LIST


In [ ]:
# ----------------------------------------------------------
# Step 7: Test 6 - Comparing Pandas with DuckDB
# ----------------------------------------------------------
import pandas as pd
from pathlib import Path

# Run this quick automated example.
import duckdb
from icare_risk.clinphen import DuckDBSource
from icare_risk.clinphen.config.schema import load_schema_from_yaml
from icare_risk.clinphen.registry.registry import load_phenotypes_from_yaml
from icare_risk.utils.paths import get_path

# Define DATA_DIR (adjust path if needed)
DATA_DIR = get_path('data/synthetic/10000p')

# ---------------------------------------------------
# 1. Traditional Pandas Loading & Date Parsing
# ---------------------------------------------------
df_episodes_pd = pd.read_parquet(DATA_DIR / "icare_episodes_anon.parquet")
df_problems_pd = pd.read_parquet(DATA_DIR / "icare_problems_anon.parquet")
df_diagnoses_pd = pd.read_parquet(DATA_DIR / "icare_episodes_diagnosis_anon.parquet")
df_prescribing_pd = pd.read_parquet(DATA_DIR / "icare_pharmacy_prescribing_anon.parquet")

df_episodes_pd["ADMISSION_DATE"] = pd.to_datetime(df_episodes_pd["ADMISSION_DATE"])
df_problems_pd["PROBLEM_DT_TM"] = pd.to_datetime(df_problems_pd["PROBLEM_DT_TM"])
df_diagnoses_pd["DIAGNOSIS_DATE"] = pd.to_datetime(df_diagnoses_pd["DIAGNOSIS_DATE"])
df_prescribing_pd["ORDER_DT_TM"] = pd.to_datetime(df_prescribing_pd["ORDER_DT_TM"])

# ---------------------------------------------------
# 2. New DuckDB Approach (Raw / Skip Mapping)
# ---------------------------------------------------

# A. Define configuration paths
schema_cfg = get_path('configs/schema.yaml')
phenotype_cfg = get_path('configs/phenotypes.yaml')

# B. Load schema and phenotype registry
schema = load_schema_from_yaml(schema_cfg)
registry = load_phenotypes_from_yaml(phenotype_cfg)
source = DuckDBSource(connection=duckdb.connect())

# 3. Extract global dependencies automatically from phenotypes.yaml
required_codes, required_cols = registry.get_global_dependencies()


# ---------------------------------------------------
# 2. New DuckDB Approach (Raw / Skip Mapping)
# ---------------------------------------------------
query = f"SELECT * FROM read_parquet('{schema.episodes.source}')"
df_episodes_duck = source.con.execute(query).fetchdf()
#df_episodes_duck = schema.episodes.normalize_frame(episodes_raw)

df_problems_duck = source.load_domain(
    schema.domains["problems"], skip_mapping=True,
    codes=required_codes.get("problems")
)
df_diagnoses_duck = source.load_domain(
    schema.domains["diagnosis"], skip_mapping=True,
    codes=['s626', 'r104']
)
df_prescribing_duck = source.load_domain(
    schema.domains["prescribing"], skip_mapping=True,
    codes=['Amoxicillin', 'Ibuprofen']
    #filters={
    #    "MEDICATION_NAME_SHORT": ['Amoxicillin', 'Ibuprofen']
    #}
)

# Parse dates for duckdb dataframes to ensure direct comparison capability
if "ADMISSION_DATE" in df_episodes_duck.columns:
    df_episodes_duck["ADMISSION_DATE"] = pd.to_datetime(df_episodes_duck["ADMISSION_DATE"])
if "PROBLEM_DT_TM" in df_problems_duck.columns:
    df_problems_duck["PROBLEM_DT_TM"] = pd.to_datetime(df_problems_duck["PROBLEM_DT_TM"])
if "DIAGNOSIS_DATE" in df_diagnoses_duck.columns:
    df_diagnoses_duck["DIAGNOSIS_DATE"] = pd.to_datetime(df_diagnoses_duck["DIAGNOSIS_DATE"])
if "ORDER_DT_TM" in df_prescribing_duck.columns:
    df_prescribing_duck["ORDER_DT_TM"] = pd.to_datetime(df_prescribing_duck["ORDER_DT_TM"])


# ---------------------------------------------------
# 3. Shape & Unique Value Comparison
# ---------------------------------------------------
datasets = {
    "Episodes": (df_episodes_pd, df_episodes_duck, "ENCNTR_ID"),
    "Problems": (df_problems_pd, df_problems_duck, "PROBLEM_CODE"),
    "Diagnoses": (df_diagnoses_pd, df_diagnoses_duck, "DIAGNOSIS_CODE"),
    "Prescribing": (df_prescribing_pd, df_prescribing_duck, "ORDER_CODE")
}

print(f"{'Dataset':<15} | {'Pandas Shape':<15} | {'DuckDB Shape':<15} | {'Unique ID Match':<15}")
print("-" * 70)

for name, (pd_df, duck_df, key_col) in datasets.items():
    # 1. Check unique value counts of the key column
    if key_col in pd_df.columns and key_col in duck_df.columns:
        pd_unique_count = pd_df[key_col].nunique()
        duck_unique_count = duck_df[key_col].nunique()
        unique_match = pd_unique_count == duck_unique_count
    else:
        unique_match = "N/A"

    # 2. Check if column names/sets match identically
    pd_cols = set(pd_df.columns)
    duck_cols = set(duck_df.columns)
    cols_match = pd_cols == duck_cols

    print(f"{name:<12} | {str(pd_df.shape):<14} | {str(duck_df.shape):<14} | {str(unique_match):<14} | {str(cols_match):<10}")

    # Optional: Detail discrepancies if columns don't match
    if not cols_match:
        missing_in_duck = pd_cols - duck_cols
        extra_in_duck = duck_cols - pd_cols
        if missing_in_duck:
            print(f"  -> Missing in DuckDB: {sorted(list(missing_in_duck))}")
        if extra_in_duck:
            print(f"  -> Extra in DuckDB:   {sorted(list(extra_in_duck))}")


✅ Schema validation passed! All files exist and all physical columns match successfully.
Skipping 'definitions': missing module or function definition.
Dataset         | Pandas Shape    | DuckDB Shape    | Unique ID Match
----------------------------------------------------------------------
Episodes     | (10000, 11)    | (10000, 13)    | N/A            | False     
  -> Missing in DuckDB: ['ADMISSION_DATE', 'ADMISSION_TIME', 'DISCHARGE_DATE', 'ENCNTR_ID', 'SUBJECT']
  -> Extra in DuckDB:   ['admission_date', 'admission_time', 'discharge_date', 'encntr', 'index_admission', 'index_discharge', 'subject']
Problems     | (30012, 6)     | (2293, 6)      | False          | True      
Diagnoses    | (29962, 12)    | (81, 12)       | N/A            | True      
Prescribing  | (69621, 16)    | (20182, 16)    | N/A            | True      


In [ ]:
# ----------------------------------------------------------
# Step 8: Reusable function
# ----------------------------------------------------------
import duckdb
from icare_risk.clinphen import DuckDBSource
from icare_risk.clinphen.io.loaders import load_pipeline_datasets
from icare_risk.clinphen.config.schema import load_schema_from_yaml
from icare_risk.clinphen.registry.registry import load_phenotypes_from_yaml
from icare_risk.utils.paths import get_path

# A. Define configuration paths
schema_cfg = get_path('configs/schema.yaml')
phenotype_cfg = get_path('configs/phenotypes.yaml')

# B. Initialize infrastructure
schema = load_schema_from_yaml(schema_cfg)
registry = load_phenotypes_from_yaml(phenotype_cfg)
source = DuckDBSource(connection=duckdb.connect())

# C. Load ONLY the domains you need.
dataframes = load_pipeline_datasets(
    source=source,
    schema=schema,
    registry=registry,
    domains_to_load=[
        "episodes", "problems", "diagnosis" # Select what you want!
    ], 
    skip_mapping=True,
    custom_codes={
        "diagnosis": ['s626', 'r104']  # Optional runtime override if needed
    }
)

# Access your loaded dataframes instantly
df_episodes_duck = dataframes.get("episodes")
df_problems_duck = dataframes.get("problems")
df_diagnoses_duck = dataframes.get("diagnosis")

print(f"Loaded successfully!")
print(f"Episodes shape: {df_episodes_duck.shape}")
print(f"Problems shape: {df_problems_duck.shape}")

✅ Schema validation passed! All files exist and all physical columns match successfully.
Skipping 'definitions': missing module or function definition.
Loaded successfully!
Episodes shape: (10000, 13)
Problems shape: (2293, 6)
